# | default_exp Testing Polars for EDA, migrating cleaned data to Vespa

In [ ]:
# | hide
# import adbc_driver_postgresql.dbapi
# from datetime import datetime
# from enum import Enum
import json
import polars as pl
# import requests
from sqlalchemy import create_engine
from sqlalchemy.orm import sessionmaker
import vespa
from vespa.deployment import VespaDocker
from vespa.io import VespaResponse, VespaQueryResponse

In [ ]:
# make sure postgres is running!
# load postgres credentials
postgres_key_path = "../secrets/postgres_login.json"
with open(postgres_key_path, "r") as fo:
    postgres_key = json.loads(fo.read())
    user = postgres_key["user"]
    password = postgres_key["password"]
    host = postgres_key["host"]


In [ ]:
# send dataframe back into Postgres
cleaned_data_uri = f"postgresql://{user}:{password}@{host}/mealeon"

query = """
    SELECT
        *
    FROM cleaned_recipes
"""

df = pl.read_database_uri(query, cleaned_data_uri, engine="adbc")

In [ ]:
print(df)

shape: (128_736, 11)
┌───────────┬──────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬───────────┐
│ mealeon_i ┆ language ┆ source_id ┆ title     ┆ … ┆ photo_url ┆ descripti ┆ steps     ┆ cuisines  │
│ d         ┆ ---      ┆ ---       ┆ ---       ┆   ┆ ---       ┆ on        ┆ ---       ┆ ---       │
│ ---       ┆ str      ┆ str       ┆ str       ┆   ┆ str       ┆ ---       ┆ list[str] ┆ list[str] │
│ str       ┆          ┆           ┆           ┆   ┆           ┆ str       ┆           ┆           │
╞═══════════╪══════════╪═══════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪═══════════╡
│ AfricanBi ┆ English  ┆ 629883    ┆ Smoked    ┆ … ┆ https://w ┆ Enjoy fal ┆ ["Remove  ┆ ["US Sout │
│ tes-3f1a4 ┆          ┆           ┆ Spatchcoc ┆   ┆ ww.africa ┆ l-off-the ┆ the       ┆ hern"]    │
│ fc7e09937 ┆          ┆           ┆ k Turkey  ┆   ┆ nbites.co ┆ -bone     ┆ giblet    ┆           │
│ 5ad…      ┆          ┆           ┆           ┆   ┆ m/w…      ┆ goodn

### Try mixing in PyVespa
Following documentation [here](https://pyvespa.readthedocs.io/en/latest/getting-started-pyvespa.html)

In [ ]:
from vespa.package import (
    ApplicationPackage,
    Field,
    Schema,
    Document,
    HNSW,
    RankProfile,
    Component,
    Parameter,
    FieldSet,
    GlobalPhaseRanking,
    Function,
    DocumentSummary,
    Summary
)

package = ApplicationPackage(
    name="mealeon3",
    schema=[
        Schema(
            name="mealeon3",
            document=Document(
                fields=[
                    Field(
                        name="language", 
                        type="string", 
                        indexing=["set_language"],
                        # match=["word"]
                    ),
                    Field(
                        name="id",
                        type="string",
                        indexing=["attribute", "summary"],
                        match=["word"],
                        bolding=True,
                    ),
                    Field(
                        name="title",
                        type="string",
                        indexing=["index", "summary"],
                        index="enable-bm25",
                        match=["word"]
                    ), 
                    Field(
                        name="description",
                        type="string",
                        indexing=["index"],
                        index="enable-bm25",
                        match=["word"],
                    ),                 
                    Field(
                        name="ingredients",
                        type="array<string>",
                        indexing=["index", "attribute"],
                        # attribute="fast-search",
                        index="enable-bm25",
                        # match=["word"],
                    ),
                    Field(
                        name="steps",
                        type="array<string>",
                        indexing=["index", "attribute"],
                        index="enable-bm25"
                    ),
                    Field(
                        name="cuisines",
                        type="array<string>",
                        indexing=["attribute", "summary"],
                        # index="enable-bm25",
                        match=["word"],
                        rank="filter"
                    ),
                    # Field(
                    #     name="embedding",
                    #     type="tensor<float>(x[384])",
                    #     indexing=[
                    #         'input title . " " . input body',
                    #         "embed",
                    #         "index",
                    #         "attribute",
                    #     ],
                    #     ann=HNSW(distance_metric="angular"),
                    #     is_document_field=False,
                    # ),
                ]
            ),
            fieldsets=[
                FieldSet(
                    name="default", 
                    fields=["title", "ingredients"]
                )
            ],
            document_summaries=[
                    DocumentSummary(
                    name="document-summary",
                    summary_fields=[
                        Summary("id")
                    ]
                ),
            ],
            rank_profiles=[
                RankProfile(
                    name="default",
                    first_phase="nativeRank(title, ingredients)"
                ),
                RankProfile(
                    name="bm25",
                    inherits="default",
                    first_phase="bm25(title) + bm25(ingredients)",
                    # inputs=[("query(q)", "tensor<float>(x[384])")],
                    functions=[
                        Function(name="bm25sum", expression="bm25(title) + bm25(ingredients)")
                    ],
                ),
                RankProfile(
                    name="combined", 
                    inherits="default",
                    first_phase="bm25(title) + bm25(ingredients) + nativeRank(title) + nativeRank(ingredients)",
                    functions=[
                        Function(name="bm25nativeRank",
                                 expression="bm25(title) + bm25(ingredients) + nativeRank(title) + nativeRank(ingredients)")
                    ]
                )
                # RankProfile(
                #     name="semantic",
                #     inputs=[("query(q)", "tensor<float>(x[384])")],
                #     first_phase="closeness(field, embedding)",
                # ),
                # RankProfile(
                #     name="fusion",
                #     inherits="bm25",
                #     inputs=[("query(q)", "tensor<float>(x[384])")],
                #     first_phase="closeness(field, embedding)",
                #     global_phase=GlobalPhaseRanking(
                #         expression="reciprocal_rank_fusion(bm25sum, closeness(field, embedding))",
                #         rerank_count=1000,
                #     ),
                # ),
            ],
        )
    ],
    # components=[
    #     Component(
    #         id="e5",
    #         type="hugging-face-embedder",
    #         parameters=[
    #             Parameter(
    #                 "transformer-model",
    #                 {
    #                     "url": "https://github.com/vespa-engine/sample-apps/raw/master/simple-semantic-search/model/e5-small-v2-int8.onnx"
    #                 },
    #             ),
    #             Parameter(
    #                 "tokenizer-model",
    #                 {
    #                     "url": "https://raw.githubusercontent.com/vespa-engine/sample-apps/master/simple-semantic-search/model/tokenizer.json"
    #                 },
    #             ),
    #         ],
    #     )
    # ],
)

In [ ]:
# try mixing in PyVespa
vespa_docker = VespaDocker(port=8183,
                           cfgsrv_port=19092)
app = vespa_docker.deploy(application_package=package)


Waiting for configuration server, 0/300 seconds...
Waiting for configuration server, 5/300 seconds...
Using plain http against endpoint http://localhost:8183/ApplicationStatus
Waiting for application status, 0/300 seconds...
Using plain http against endpoint http://localhost:8183/ApplicationStatus
Waiting for application status, 5/300 seconds...
Using plain http against endpoint http://localhost:8183/ApplicationStatus
Waiting for application status, 10/300 seconds...
Using plain http against endpoint http://localhost:8183/ApplicationStatus
Waiting for application status, 15/300 seconds...
Using plain http against endpoint http://localhost:8183/ApplicationStatus
Waiting for application status, 20/300 seconds...
Using plain http against endpoint http://localhost:8183/ApplicationStatus
Waiting for application status, 25/300 seconds...
Using plain http against endpoint http://localhost:8183/ApplicationStatus
Application is up!
Finished deployment.


In [ ]:
# query should be recipe name?
    # WHERE title !contains {query}
# cuisine name should be in the WHERE filter clause of YQL
    # AND WHERE cuisine NOT IN {cuisines}
# how to penalize similar title?

# start with plain keyword search

with app.syncio(connections=1) as session:
    query = "Buffalo Wings"
    response: VespaQueryResponse = session.query(
        yql=f"select * from sources mealeon3 where (title contains '{query}') limit 10",
        query=query,
        ranking="bm25"
        # body={"input.query(q)": f"embed({query})"},
    )
    assert response.is_successful()

In [ ]:
print(response.hits)

[]


In [ ]:
import pprint

pprint.pp(response.hits)

[]


In [ ]:
# list of recipe IDs
baseline_to_search = [
    'BBCFood-11015de4db20e53dcda0035673dfdbeca355d2a1d37d71f13d5fe5ea7ccca02d',
    'Epicurious-32b7b2040aba4e44b0f5e34b094aafc6b1b7168e689ba6c92428fa8c9e235c5a',
    'Epicurious-9e24d01c507b4ee64ee484fc4efafb5f3d44709abb8fc36f1b34a91f7b69b678',
    'AllRecipes-fddb273d401ac45b568695daeb1abe0b632696dd867992fd401e31dc794d66f9',
    'AllRecipes-715d84d02e183deaaf2400f7526c80a2b01a25032c1abb125c28fe4d879cc672', 
    'AfricanBites-6f62a39913dd8029cd399d4ae47b0551f94a16865a3143b5dd28664c0880596e', 
    'Epicurious-85d1faf0a617f34521543d7c64122288b8ba65ae55b7a37a9abeb22d8845ff11',
    'AllRecipes-867b4f3b539b381fb9f35252402fc14efc9d6fcc64353089959ac159372a8199', 
    'BBCFood-c6b9581aa49ee6b0cc3f686f58cfd124c395284cc46dfcde5bc931982975ee15',
    'BBCFood-d59ab715a7f6f0022155b1149e4e059b0497fd790993fb11f1795b901cae8f97',
    'AllRecipes-d431cefd9f595be8a8fcd7ff9e05a82ca1b79220c72edde1aa2a0b6574060c04',
    'BBCFood-056f016b0b0d8c383c1a6bf47077a46a64529c04ab043c14dce3b23d3980e0ed',
    'AllRecipes-d4bb95742a7fd71eb848a7d4b2a4dc13220e5bb378e1b6805d5ea493e0e1166d',
    'Epicurious-0293b95bfc74cf90dc07e2ee399312e40e03d14dd99e5db84e678a1393b131cc',
    'Panlasang_Pinoy-1d186c4a4c274e42580488e652e3448aa9942642fa9e990d89d3e1894056ce78',
    'AllRecipes-9bfd691bb0521c3bba0275d1ef18ac30ea15c2aeb0a48f2b9df1d5defd962bed',
    'Epicurious-e7efaeb37ea032d7faaad76c1e0675f987aa5eb162e5447927ae510c0301807e',
    'Epicurious-ca2d5e56121ae04e8b7bdcd07648796b2d95afed017fc56200535ad43ca8e51d',
    'Epicurious-7761b0f03aaf56b52b1de417213b66fddf1d49c554dabaacb23c9c7a260982ba',
    'Epicurious-e367b68ed6df8b0bccbeb73f9d82897294ef267fa3a74d30505ce7470b2384dc',
    'AllRecipes-a0ae2dfd0ca25ae3146df101d1dc0e7c54e39eec69617ff634f42d80d7b70be4',
    'AfricanBites-8e8956de6dd17820c4e5146820972193f2587d97e80ee1a754c92dce048e4f9e',
    'BBCFood-3aa9507e04707431ec512662444ef602910c6c0ecc997de7ea206729985c5c08',
    'AllRecipes-6914cfedf5d3af52ea66433959c958821b8940ee1948fbe29f841d48ad1de4a4',
    'AfricanBites-2f35554786f85a304f1f5e86313f5d35730777a4f0861edf3474a914e9b7ec23',
    'Epicurious-0a033ff3220eae4b638fd08de58646ba84ed5f184b89d792afe2ed90a8c4a859',
    'BBCFood-adcbfceaa8636f83ff8866d44bc7a44385c71250d73d427c08dbe345e4127143',
    'AllRecipes-81a545cc3eac6e8b89274a6a9f4a6574232e40c3917f499a431d9fa6ebc540bf',
    'BBCFood-0a7c810d7b7c99ae728f9a2deff14340edb099949f20fed75a8083c29ca83aab',
    'Epicurious-a0ab4c8e25f1fe8e65f400f93db23cf55e1e46cb4361cd65426f908f3013f33d',
    'AllRecipes-bb6dd1b85cfd3cbfeb68160c187862ec6de84739123733ceeaae2976141f8936',
    'Epicurious-3b409b8e753718bdb272130405491c4dcb4c26e73412b54cea408319dc932c6e',
    'AllRecipes-92c5c30f5c0b1655a1e1900fcdaa603917f2abf62529c130713412a844046e33',
    'AfricanBites-fc015a5953d8811734f652ad17ba50d73f98d783350df0a95a6e2015007263f0',
    'AllRecipes-384fd2f67b06a5bfdf532f94d5e89df99e08b91c5d9dace20fd29bba7b8fdad3',
    'Epicurious-aa7806301b6fc904134d692a2a425d0a0fd5a1f6f95cb431e8c31173db7d99fa',
    'AllRecipes-d9b8ecb976729402694fa448a53b741404ef3e10f6dd9fe2219fc6fb3fb7b038',
    'BBCFood-ee117ce84728c9b2e90654f59895a89dc7c10599b61bed83f55e9bdf723e6890',
    'BBCFood-53dc7bd8da3ef8a7dedfef0b2be1880f71585752d8113313c6d9e49bd5cba6fa',
    'BBCFood-2ab22cd0a3889f38cf740040d33aae5a4baa8568b70cde5233d6f8a673958140',
    'Epicurious-1c0fe6496b26a8b7bdf549e9ed70bc9802d7df404b89f1d3de6d01a1a44d31f8',
    'BBCFood-47410dc7097b2609d5c0a80e4ce6f938b00be18b52907e5a5322e33d6edee335',
    'AfricanBites-587700762275fbf6a6e2a343b27cc689317725ae2d1724cd33c6c841583afc3e',
    'Epicurious-ca99be627e88806d8357a5cbfdd314b3643028f275e6ae1c90adfcf3ae9e30d4',
    'Epicurious-7e9ff3fc13b040941ed9f7a4bbfd809e17a9ce7d5b569f1f6c4ba96e9a9ae140',
    'Epicurious-4b6f818877199d375e801cec974c6f6b037b4c0b2aa2bbec0238dd77f797264d',
    'AllRecipes-c6d0cd77e37ed8069f0d9e72f9e832026459273ac1bc2587800dc8fcd1418ec7',
    'AllRecipes-48bde26e015dc9cbcbb540c4710248d538227d4afe5aa3e977e34710289789bb',
    'AllRecipes-cac0d29872bd9f49d34d8c9eac41a3806929420f84e36b8e579c3bce939cdc7b',
    'AllRecipes-9a5dc3fa2511284deeb5ee469d110874b0de9c021439cd0e088d3ef4f53f28bf',
    'BBCFood-6d6cc18dadaffa5ea10194b59aa6e310d34b1f713756d0b528cc57c5e61d92e3',
    'AllRecipes-5fc2fc72d7caf510b4ccde625c54a0907de1489d323d385d74c2a6c9c5d61c7b',
    'AfricanBites-56e7fc4d3a1db457444cf4744911356514b3ac961f04976a0fb127f0ac8145be',
    'AllRecipes-8aa892f3f4d827f781d2bc1491351d982144e2fa2bb3226ebf5cc6af6df5ac5a',
    'AllRecipes-e13da6adb836dcf514a8f285e162a9529271a647a11f2da6e60697541260c1ec',
    'Epicurious-aef6e4737b2b9248ae51893ec6d2295ab2380bc67e817ebef13416c65d44a3a6',
    'AllRecipes-e11c702c9c5f8d5c0d63a6b58a60d90b87f1f4e88dad3586b91b3248bb721504',
    'Epicurious-b6ae682f94cb0c4bf256ebfc569550702ef82612457ff8e601cf9e61e0b5f4ab',
    'Epicurious-9908959a4069c75bd1d95c4b11678d15a63a153c91d164d3f7d1159d35ae05dc',
    'AllRecipes-4f31ce862b67de4659f76e3a4de016a32770c878544d112aae95cc09b63d799a',
    'AllRecipes-f3aafa676ce976137a1e9a62a6beeca6288c2a4b0c35ba2c4da9cd6a4fa9f466',
    'Epicurious-b3f8d44e48bba1b47e5cf1482d8c5e076359fa919061bbfd2f075d131aea2025',
    'Epicurious-6a7b3ef6808832fbf1ab5a8e539eb794866d630c1b65218ee7b5c385b78f65a4',
    'AllRecipes-0caa9e3e35dee2910e6f81c869d1176a2de129b8e3c7fddf6e469b9e3977fadb',
    'AllRecipes-3c9cbb4d78d017a21a48907adbc31c2593ebbd0717b9391779a2f75b531fcbe4',
    'AllRecipes-3e906cbcfdee7e4c52d087efc24aea9c7e3b760a8c5f0d1c85c459f3354b643d',
    'Panlasang_Pinoy-b5a6e813375f4cbb399a2e68dec8f9448d64c4f01f2e5b5accd9188509875f17',
    'Epicurious-d2b862c91c84a69b9b2f2be1a1cc9702a2d6af8a6f1e5dcaccbacda6ed9f31b0',
    'AllRecipes-9e94827f263b0409c124dd60e95e4a61e1d446940e3ad3b88b4e3b43d00b6f65',
    'AllRecipes-b8116cc920cd9d916cd196927ad2af1062785cdaa3d3b4fc3e997ee86fd75c84',
    'Epicurious-4284e53129c7dc89f97f2f1933843946745df061370bd1ab738bf2afe1c537c3',
    'AllRecipes-2b9df9cae8603bc9f0c5ddadf04fe529c1174d38a7e32031e42ee9897ea8e05f',
    'AfricanBites-5ca4ea56e8d49013ebd7457560a386a5e1a1059f267514b7c1dff7b9f0c78d72',
    'AllRecipes-bd864502bd1b052f501d3148d66effc966246a69a5e705c585fa366d9599e090',
    'AllRecipes-1b1ac4f279d0cbb0c680da7a1839d0031a52f3cc73681dbb2644cf96d6aec4f9'
]

In [ ]:
filtered_to_investigate = df.filter(pl.col("mealeon_id").is_in(baseline_to_search))

In [ ]:
filtered_to_investigate

mealeon_id,language,source_id,title,origin,url,ingredients,photo_url,description,steps,cuisines
str,str,str,str,str,str,list[str],str,str,list[str],list[str]
"""AfricanBites-587700762275fbf6a…","""English""","""597818""","""Banana Pudding""","""AfricanBites""","""https://www.africanbites.com/h…","[""2 ½ tablespoon s (18.75 g) cornstarch "", ""6 tablespoons (75 g) granulated sugar "", … ""homemade whipped cream ""]","""https://www.africanbites.com/w…","""Homemade Banana Pudding – a cl…","[""Add dry ingredients to a saucepan: cornstarch, sugar and salt. Mix to combine. "", ""In a medium bowl, whisk together milk, heavy cream, and egg yolks. Gently whisk wet ingredients into the saucepan making sure there are no lumps or until sugar has dissolved . "", … ""Serve or refrigerate until ready to eat. ""]","[""US Southern""]"
"""AfricanBites-6f62a39913dd8029c…","""English""","""598753""","""Tiramisu""","""AfricanBites""","""https://www.africanbites.com/t…","[""5 large egg yolks , room temperature"", ""¾ cup granulated sugar"", … ""1 cup or more homemade whipped cream (optional, for piping) ""]","""https://www.africanbites.com/w…","""This classic Italian coffee-fl…","[""In a medium saucepan, whisk together egg yolks and sugar until combined. Whisk in milk and cook over medium/low heat stirring constantly. Bring to a boil and boil for 1 minute. Remove from heat and allow to cool. Cover with plastic wrap and refrigerate for 1 hour. "", ""Whisk mascarpone cheese into yolk mixture until smooth. "", … ""Slice or scoop to serve.""]","[""Italian""]"
"""AfricanBites-fc015a5953d881173…","""English""","""607516""","""Apple Pie""","""AfricanBites""","""https://www.africanbites.com/a…","[""Homemade pie crust or store-bought pie crust "", ""7-8 apples (about 3 ½ pounds)"", … ""coarse sugar or turbinado sugar for sprinkling""]","""https://www.africanbites.com/w…","""Made with lightly spiced apple…","[""Start by making the pie crust/and or preparing the pie pan. "", ""Roll out pie crust"", … ""Bake for 40 to 45 minutes or until the pie is bubbly and the crust is golden brown. Remove the foil on the last 15 minutes before the pie has cooked through. Serve warm with ice cream. ""]","[""American""]"
"""AfricanBites-2f35554786f85a304…","""English""","""608647""","""Fish and Chips""","""AfricanBites""","""https://www.africanbites.com/f…","[""1 ¼ cup (5 ounces) all-purpose flour"", ""¾ cup (4 ounces) cornstarch"", … ""salt and seasonings to taste""]","""https://www.africanbites.com/w…","""Perfectly seasoned tender fish…","[""In a large bowl, whisk together flour, cornstarch, garlic powder, cayenne, paprika, baking powder, and salt and pepper, to taste. Reserve about ¾ cup of the flour mixture (this is used to dust fish). Set aside until ready to dip the fish. "", ""Pour oil into a cast iron or skillet, enough for deep frying. Preheat oil to about 375 degrees F."", … ""Toss the fries with seasonings or parmesan cheese, if desired, and serve with fish.""]","[""British""]"
"""AfricanBites-5ca4ea56e8d49013e…","""English""","""567371""","""Huli Huli Chicken""","""AfricanBites""","""https://www.africanbites.com/h…","[""3 1/2- 4 pounds chicken drumsticks"", ""1 cup unsweetened pineapple juice"", … ""Green onions sliced for garnish""]","""https://www.africanbites.com/w…","""Huli Huli Chicken - Grilled or…","[""In a medium bowl, mix the pineapple juice, soy sauce, honey, brown sugar, sriracha, ketchup, ginger, cumin, lemon juice and garlic.""]","[""American""]"
…,…,…,…,…,…,…,…,…,…,…
"""AllRecipes-9e94827f263b0409c12…","""English""","""14713""","""Souvlaki""","""AllRecipes""","""https://www.allrecipes.com/rec…","[""¼ cup olive oil"", ""¼ cup soy sauce"", … "" skewers""]","""https://www.allrecipes.com/thm…","""Souvlaki is a Greek specialty …","[""This homemade souvlaki recipe is flavorful, tender, and easy to make on your outdoor grill."", ""Souvlaki is a Greek fast food that consists of meat grilled on a skewer. It can be served alone or inside of a rolled pita. It’

First things we need to do are check that these recipes "make sense"
- Are these recipes good baselines?

In [ ]:
with pl.Config(fmt_table_cell_list_len=20,
               fmt_str_lengths=1000,
               set_tbl_width_chars=70
               ):
    print(filtered_to_investigate.select(["title", "ingredients", "description", "cuisines"])[0:5])

shape: (5, 4)
┌─────────────────┬────────────────┬────────────────┬────────────────┐
│ title           ┆ ingredients    ┆ description    ┆ cuisines       │
│ ---             ┆ ---            ┆ ---            ┆ ---            │
│ str             ┆ list[str]      ┆ str            ┆ list[str]      │
╞═════════════════╪════════════════╪════════════════╪════════════════╡
│ Banana Pudding  ┆ ["2 ½          ┆ Homemade       ┆ ["US           │
│                 ┆ tablespoon s   ┆ Banana Pudding ┆ Southern"]     │
│                 ┆ (18.75 g)      ┆ – a classic    ┆                │
│                 ┆ cornstarch ",  ┆ Southern       ┆                │
│                 ┆ "6 tablespoons ┆ no-bake        ┆                │
│                 ┆ (75 g)         ┆ dessert with   ┆                │
│                 ┆ granulated     ┆ decadent       ┆                │
│                 ┆ sugar ",       ┆ layers of      ┆                │
│                 ┆ "Pinch of salt ┆ vanilla        ┆          

Let's re-apply the previous text processing on ingredients to shorten these huge lists and make them easier to read (previous text processing removed numbers and I think units)

Previously, using `TFIDFVectorizer`, the params were:
```
TfidfVectorizer(
    stop_words=stopwords_list,
    min_df=2,
    token_pattern=r"(?u)\b[a-zA-Z]{2,}\b",
    preprocessor=lemmatizer.lemmatize,
)
```

Stopwords were in a CSV located at `../food_stopwords.csv` and did include units and brands

Probably do not need to lemmatize, just apply that regex from token_pattern and filter out stopwords.

How to reapply in Polars?
- [X] Join list of strings (ingredients)
- [X] filter string based on regex and stopwords

In [ ]:
filtered_to_investigate.select("ingredients").to_series().list.join(separator="||")

ingredients
str
"""2 ½ tablespoon s (18.75 g) co…"
"""5 large egg yolks , room tempe…"
"""Homemade pie crust or store-bo…"
"""1 ¼ cup (5 ounces) all-purpo…"
"""3 1/2- 4 pounds chicken drumst…"
…
"""¼ cup olive oil||¼ cup soy sau…"
"""1 ½ cups all-purpose flour||¾ …"
"""2 onions, finely chopped||3 c…"


In [ ]:
filtered_to_investigate = filtered_to_investigate.with_columns(
    pl.col("ingredients")
    .list.join(separator="BREAK")
    .str.extract_all(
        r"(?u)\b[a-zA-Z]{2,}\b"
    )
    .alias("regex_filtered_ingredients")
)

In [ ]:
filtered_to_investigate.glimpse()

Rows: 68
Columns: 12
$ mealeon_id                       <str> 'AfricanBites-587700762275fbf6a6e2a343b27cc689317725ae2d1724cd33c6c841583afc3e', 'AfricanBites-6f62a39913dd8029cd399d4ae47b0551f94a16865a3143b5dd28664c0880596e', 'AfricanBites-fc015a5953d8811734f652ad17ba50d73f98d783350df0a95a6e2015007263f0', 'AfricanBites-2f35554786f85a304f1f5e86313f5d35730777a4f0861edf3474a914e9b7ec23', 'AfricanBites-5ca4ea56e8d49013ebd7457560a386a5e1a1059f267514b7c1dff7b9f0c78d72', 'AfricanBites-56e7fc4d3a1db457444cf4744911356514b3ac961f04976a0fb127f0ac8145be', 'AfricanBites-8e8956de6dd17820c4e5146820972193f2587d97e80ee1a754c92dce048e4f9e', 'Panlasang_Pinoy-1d186c4a4c274e42580488e652e3448aa9942642fa9e990d89d3e1894056ce78', 'Panlasang_Pinoy-b5a6e813375f4cbb399a2e68dec8f9448d64c4f01f2e5b5accd9188509875f17', 'AllRecipes-d9b8ecb976729402694fa448a53b741404ef3e10f6dd9fe2219fc6fb3fb7b038'
$ language                         <str> 'English', 'English', 'English', 'English', 'English', 'English', 'English', 'Englis

In [ ]:
# lets load the food_stopwords.csv into a polars dataframe and extract into a set
with open("../food_stopwords.txt") as f:
    for line in f:
        food_stopwords = line.replace('"', '').split(",")

food_stopwords

['spare',
 'pernod',
 'snow',
 'hollowed',
 'bonein',
 'for',
 'pans',
 'knorr',
 'eighths',
 'approximately',
 'kitchen',
 'canner',
 'crosswise',
 'use',
 'fully',
 'parts',
 'best®',
 'filtered',
 'freshepazote',
 'tasting',
 'flavoring',
 'blended',
 'plump',
 'gold®',
 'percent',
 'insert',
 'shreds',
 'smaller',
 'wiped',
 'ragu',
 'preparation',
 'thirty',
 'king',
 'hinged',
 'half',
 'needed',
 'elixir',
 'starter',
 'nestle',
 'naturally',
 'leafed',
 'attachment',
 'leaved',
 'foil',
 'ROTEL',
 'karo',
 'omitting',
 'with',
 'cracks',
 'cube',
 'precooked',
 'scrubbed',
 'coins',
 'area',
 'earth balance',
 'johnsonville',
 'farm',
 'flavor',
 'online',
 'country bob',
 'jumbo',
 'sticky',
 'crescent',
 'eighteen',
 'boil',
 'ready',
 'superfine',
 'assorted',
 'imported',
 'style',
 'micro',
 'angelhair',
 'preserved',
 'freeze',
 'clark',
 'hon',
 'ripe',
 'grated',
 'jellied',
 'malt-o-meal',
 'cups',
 'handheld',
 'platters',
 'tree',
 'unsalted',
 'store-bought',
 'pick

In [ ]:
filtered_to_investigate = filtered_to_investigate.with_columns(
    pl.col("ingredients")
    .list.join(separator="|")
    .str.to_lowercase()
    .str.extract_all(
        r"(?u)\b[a-zA-Z]{2,}\b"
    )
    .list.set_difference(food_stopwords)
    .alias("regex_filtered_ingredients")
)

In [ ]:
filtered_to_investigate.select("regex_filtered_ingredients").glimpse()

Rows: 68
Columns: 1
$ regex_filtered_ingredients <list[str]> ['cream', 'cornstarch', 'vanilla', 'milk', 'sugar', 'bananas', 'extract', 'salt', 'egg', 'butter', 'wafers'], ['liquor', 'egg', 'out', 'rum', 'wine', 'ladyfingers', 'vanilla', 'sugar', 'cream', 'milk', 'mascarpone', 'cheese', 'espresso', 'cocoa', 'alcohol', 'marsala', 'coffee', 'liqueur', 'extract'], ['sprinkling', 'pie', 'turbinado', 'butter', 'salt', 'milk', 'apples', 'ginger', 'cream', 'nutmeg', 'brown', 'sugar', 'if', 'cornstarch', 'lemon', 'cinnamon', 'allspice', 'egg'], ['russet', 'vinegar', 'water', 'beer', 'flour', 'cornstarch', 'potatoes', 'garlic', 'fillet', 'ice', 'cayenne', 'pepper', 'paprika', 'baking', 'salt', 'white', 'cod'], ['sriracha', 'chicken', 'drumsticks', 'lemon', 'spice', 'pineapple', 'broth', 'soy', 'sauce', 'brown', 'sugar', 'garlic', 'honey', 'ketchup', 'tablsespoon', 'onions', 'cumin', 'ginger', 'adjust', 'green'], ['jalapenos', 'parsley', 'rice', 'replace', 'basmati', 'chili', 'onion', 'if', 'wate

Look at recipes, see if ingredients make sense with a canonical version online

In [ ]:
with pl.Config(fmt_table_cell_list_len=20,
               fmt_str_lengths=1000,
               set_tbl_width_chars=70
               ):
    print(filtered_to_investigate.select(["title", "mealeon_id","regex_filtered_ingredients"])[0:5])


shape: (5, 3)
┌───────────────────┬────────────────────────┬───────────────────────┐
│ title             ┆ mealeon_id             ┆ regex_filtered_ingred │
│ ---               ┆ ---                    ┆ ients                 │
│ str               ┆ str                    ┆ ---                   │
│                   ┆                        ┆ list[str]             │
╞═══════════════════╪════════════════════════╪═══════════════════════╡
│ Banana Pudding    ┆ AfricanBites-587700762 ┆ ["cream",             │
│                   ┆ 275fbf6a6e2a343b27cc68 ┆ "cornstarch",         │
│                   ┆ 9317725ae2d1724cd33c6c ┆ "vanilla", "milk",    │
│                   ┆ 841583afc3e            ┆ "sugar", "bananas",   │
│                   ┆                        ┆ "extract", "salt",    │
│                   ┆                        ┆ "egg", "butter",      │
│                   ┆                        ┆ "wafers"]             │
│ Tiramisu          ┆ AfricanBites-6f62a3991 ┆ ["liquor", "egg"

In [ ]:
with pl.Config(fmt_table_cell_list_len=20,
               fmt_str_lengths=1000,
               set_tbl_width_chars=70
               ):
    print(filtered_to_investigate.select(["title", "mealeon_id","regex_filtered_ingredients"])[5:10])


shape: (5, 3)
┌──────────────────┬────────────────────────┬────────────────────────┐
│ title            ┆ mealeon_id             ┆ regex_filtered_ingredi │
│ ---              ┆ ---                    ┆ ents                   │
│ str              ┆ str                    ┆ ---                    │
│                  ┆                        ┆ list[str]              │
╞══════════════════╪════════════════════════╪════════════════════════╡
│ Mexican Rice     ┆ AfricanBites-56e7fc4d3 ┆ ["jalapenos",          │
│                  ┆ a1db457444cf4744911356 ┆ "parsley", "rice",     │
│                  ┆ 514b3ac961f04976a0fb12 ┆ "replace", "basmati",  │
│                  ┆ 7f0ac8145be            ┆ "chili", "onion",      │
│                  ┆                        ┆ "if", "water",         │
│                  ┆                        ┆ "spice", "tomatoes",   │
│                  ┆                        ┆ "sauce", "kids",       │
│                  ┆                        ┆ "black", "broth",

In [ ]:
with pl.Config(fmt_table_cell_list_len=20,
               fmt_str_lengths=200,
               set_tbl_width_chars=50
               ):
    print(filtered_to_investigate.select(["title", "mealeon_id","regex_filtered_ingredients"])[10:15])


shape: (5, 3)
┌──────────────┬────────────────┬────────────────┐
│ title        ┆ mealeon_id     ┆ regex_filtered │
│ ---          ┆ ---            ┆ _ingredients   │
│ str          ┆ str            ┆ ---            │
│              ┆                ┆ list[str]      │
╞══════════════╪════════════════╪════════════════╡
│ Tiramisu     ┆ AllRecipes-867 ┆ ["cookies",    │
│              ┆ b4f3b539b381fb ┆ "egg",         │
│              ┆ 9f35252402fc14 ┆ "cheese",      │
│              ┆ efc9d6fcc64353 ┆ "rum",         │
│              ┆ 089959ac159372 ┆ "white",       │
│              ┆ a8199          ┆ "sugar",       │
│              ┆                ┆ "milk",        │
│              ┆                ┆ "ladyfinger",  │
│              ┆                ┆ "cocoa",       │
│              ┆                ┆ "cream",       │
│              ┆                ┆ "coffee",      │
│              ┆                ┆ "vanilla",     │
│              ┆                ┆ "extract",     │
│              ┆ 

In [ ]:
with pl.Config(fmt_table_cell_list_len=50,
               fmt_str_lengths=2000,
               set_tbl_width_chars=50,
               ):
    print(filtered_to_investigate.select(["title", "mealeon_id","regex_filtered_ingredients"])[15:20])


shape: (5, 3)
┌────────────────┬───────────────┬───────────────┐
│ title          ┆ mealeon_id    ┆ regex_filtere │
│ ---            ┆ ---           ┆ d_ingredients │
│ str            ┆ str           ┆ ---           │
│                ┆               ┆ list[str]     │
╞════════════════╪═══════════════╪═══════════════╡
│ Cannoli        ┆ BBCFood-d59ab ┆ ["orange",    │
│                ┆ 715a7f6f00221 ┆ "mascarpone", │
│                ┆ 55b1149e4e059 ┆ "flour",      │
│                ┆ b0497fd790993 ┆ "vegetable",  │
│                ┆ fb11f1795b901 ┆ "only",       │
│                ┆ cae8f97       ┆ "cinnamon",   │
│                ┆               ┆ "ricotta",    │
│                ┆               ┆ "chocolate",  │
│                ┆               ┆ "caster",     │
│                ┆               ┆ "sugar",      │
│                ┆               ┆ "wine", "bica │
│                ┆               ┆ rbonate",     │
│                ┆               ┆ "zest",       │
│                

| title (str) | mealeon_id (str) | reference_url (str) | passable (y/n) |
│ Banana Pudding | AfricanBites-587700762275fbf6a6e2a343b27cc689317725ae2d1724cd33c6c841583afc3e | https://www.the-girl-who-ate-everything.com/magnolia-bakerys-famous-banana-pudding/ | yes |
| Tiramisu | AfricanBites-6f62a39913dd8029cd399d4ae47b0551f94a16865a3143b5dd28664c0880596e | 
https://cooking.nytimes.com/recipes/1018684-classic-tiramisu | yes | 
| Apple Pie | AfricanBites-fc015a5953d8811734f652ad17ba50d73f98d783350df0a95a6e2015007263f0 | https://www.allrecipes.com/recipe/12682/apple-pie-by-grandma-ople/ | yes | 
│ Fish and Chips | AfricanBites-2f35554786f85a304f1f5e86313f5d35730777a4f0861edf3474a914e9b7ec23 | https://www.thespruceeats.com/best-fish-and-chips-recipe-434856 | yes | 
│ Huli Huli Chicken | AfricanBites-5ca4ea56e8d49013ebd7457560a386a5e1a1059f267514b7c1dff7b9f0c78d72 | https://www.allrecipes.com/recipe/229690/huli-huli-chicken/ │ yes │
| 

In [ ]:
with pl.Config(fmt_table_cell_list_len=50,
               fmt_str_lengths=2000,
               set_tbl_width_chars=50,
               ):
    print(filtered_to_investigate.select(
        ["title", 
         "mealeon_id",
         "regex_filtered_ingredients"
         ]
         )[20:25]
         )


shape: (5, 3)
┌───────────────┬────────────────┬───────────────┐
│ title         ┆ mealeon_id     ┆ regex_filtere │
│ ---           ┆ ---            ┆ d_ingredients │
│ str           ┆ str            ┆ ---           │
│               ┆                ┆ list[str]     │
╞═══════════════╪════════════════╪═══════════════╡
│ Haggis        ┆ BBCFood-2ab22c ┆ ["sheep",     │
│               ┆ d0a3889f38cf74 ┆ "stomach",    │
│               ┆ 0040d33aae5a4b ┆ "pepper",     │
│               ┆ aa8568b70cde52 ┆ "ox",         │
│               ┆ 33d6f8a6739581 ┆ "secum",      │
│               ┆ 40             ┆ "salt",       │
│               ┆                ┆ "nutmeg",     │
│               ┆                ┆ "onions",     │
│               ┆                ┆ "coriander",  │
│               ┆                ┆ "turned",     │
│               ┆                ┆ "oatmeal",    │
│               ┆                ┆ "out", "fat", │
│               ┆                ┆ "mace",       │
│               ┆

In [ ]:
with pl.Config(fmt_table_cell_list_len=50,
               fmt_str_lengths=2000,
               set_tbl_width_chars=50,
               ):
    print(filtered_to_investigate.select(
        ["title", 
         "mealeon_id",
         "regex_filtered_ingredients"
         ]
         )[25:30]
         )


shape: (5, 3)
┌────────────┬─────────────────┬─────────────────┐
│ title      ┆ mealeon_id      ┆ regex_filtered_ │
│ ---        ┆ ---             ┆ ingredients     │
│ str        ┆ str             ┆ ---             │
│            ┆                 ┆ list[str]       │
╞════════════╪═════════════════╪═════════════════╡
│ Margherita ┆ AllRecipes-715d ┆ ["tomatoes",    │
│ Pizza      ┆ 84d02e183deaaf2 ┆ "pam", "olive", │
│            ┆ 400f7526c80a2b0 ┆ "oil", "no",    │
│            ┆ 1a25032c1abb125 ┆ "cornmeal",     │
│            ┆ c28fe4d879cc672 ┆ "cheese",       │
│            ┆                 ┆ "plum",         │
│            ┆                 ┆ "mozzarella",   │
│            ┆                 ┆ "flour",        │
│            ┆                 ┆ "bread",        │
│            ┆                 ┆ "dough",        │
│            ┆                 ┆ "yellow",       │
│            ┆                 ┆ "basil",        │
│            ┆                 ┆ "hunt",         │
│            ┆   

In [ ]:
with pl.Config(fmt_table_cell_list_len=50,
               fmt_str_lengths=2000,
               set_tbl_width_chars=50,
               ):
    print(filtered_to_investigate.select(
        ["title", 
         "mealeon_id",
         "regex_filtered_ingredients"
         ]
         )[31:35]
         )


shape: (4, 3)
┌────────────────┬───────────────┬───────────────┐
│ title          ┆ mealeon_id    ┆ regex_filtere │
│ ---            ┆ ---           ┆ d_ingredients │
│ str            ┆ str           ┆ ---           │
│                ┆               ┆ list[str]     │
╞════════════════╪═══════════════╪═══════════════╡
│ Kung Pao       ┆ AllRecipes-9b ┆ ["paste",     │
│ Chicken        ┆ fd691bb0521c3 ┆ "cornstarch", │
│                ┆ bba0275d1ef18 ┆ "vinegar",    │
│                ┆ ac30ea15c2aeb ┆ "garlic",     │
│                ┆ 0a48f2b9df1d5 ┆ "water",      │
│                ┆ defd962bed    ┆ "white",      │
│                ┆               ┆ "wine",       │
│                ┆               ┆ "brown",      │
│                ┆               ┆ "soy",        │
│                ┆               ┆ "sauce",      │
│                ┆               ┆ "sesame",     │
│                ┆               ┆ "oil",        │
│                ┆               ┆ "sugar",      │
│                

In [ ]:
with pl.Config(fmt_table_cell_list_len=50,
               fmt_str_lengths=2000,
               set_tbl_width_chars=50,
               ):
    print(filtered_to_investigate.select(
        ["title", 
         "mealeon_id",
         "regex_filtered_ingredients"
         ]
         )[35:40]
         )


shape: (5, 3)
┌────────────────┬───────────────┬───────────────┐
│ title          ┆ mealeon_id    ┆ regex_filtere │
│ ---            ┆ ---           ┆ d_ingredients │
│ str            ┆ str           ┆ ---           │
│                ┆               ┆ list[str]     │
╞════════════════╪═══════════════╪═══════════════╡
│ Chicken Tikka  ┆ Epicurious-a0 ┆ ["garlic",    │
│ Masala         ┆ ab4c8e25f1fe8 ┆ "vegetable",  │
│                ┆ e65f400f93db2 ┆ "oil",        │
│                ┆ 3cf55e1e46cb4 ┆ "rice",       │
│                ┆ 361cd65426f90 ┆ "steamed",    │
│                ┆ 8f3013f33d    ┆ "cream",      │
│                ┆               ┆ "ginger",     │
│                ┆               ┆ "cardamom",   │
│                ┆               ┆ "turmeric",   │
│                ┆               ┆ "garam",      │
│                ┆               ┆ "masala",     │
│                ┆               ┆ "coriander",  │
│                ┆               ┆ "cumin",      │
│                

In [ ]:
with pl.Config(fmt_table_cell_list_len=50,
               fmt_str_lengths=2000,
               set_tbl_width_chars=50,
               ):
    print(filtered_to_investigate.select(
        ["title", 
         "mealeon_id",
         "regex_filtered_ingredients"
         ]
         )[41:45]
         )


shape: (4, 3)
┌──────────┬──────────────────┬──────────────────┐
│ title    ┆ mealeon_id       ┆ regex_filtered_i │
│ ---      ┆ ---              ┆ ngredients       │
│ str      ┆ str              ┆ ---              │
│          ┆                  ┆ list[str]        │
╞══════════╪══════════════════╪══════════════════╡
│ Kalbi    ┆ Epicurious-7e9ff ┆ ["bbq",          │
│          ┆ 3fc13b040941ed9f ┆ "marinade",      │
│          ┆ 7a4bbfd809e17a9c ┆ "scallions",     │
│          ┆ e7d5b569f1f6c4ba ┆ "korean",        │
│          ┆ 96e9a9ae140      ┆ "short", "ribs", │
│          ┆                  ┆ "each"]          │
│ Bagels   ┆ Epicurious-b3f8d ┆ ["salt",         │
│          ┆ 44e48bba1b47e5cf ┆ "baking",        │
│          ┆ 1482d8c5e076359f ┆ "barley",        │
│          ┆ a919061bbfd2f075 ┆ "malt", "syrup", │
│          ┆ d131aea2025      ┆ "honey",         │
│          ┆                  ┆ "flour", "rice", │
│          ┆                  ┆ "water",         │
│          ┆     

In [ ]:
with pl.Config(fmt_table_cell_list_len=50,
               fmt_str_lengths=2000,
               set_tbl_width_chars=50,
               ):
    print(filtered_to_investigate.select(
        ["title", 
         "mealeon_id",
         "regex_filtered_ingredients"
         ]
         )[45:50]
         )


shape: (5, 3)
┌────────────────┬───────────────┬───────────────┐
│ title          ┆ mealeon_id    ┆ regex_filtere │
│ ---            ┆ ---           ┆ d_ingredients │
│ str            ┆ str           ┆ ---           │
│                ┆               ┆ list[str]     │
╞════════════════╪═══════════════╪═══════════════╡
│ Pierogies      ┆ Epicurious-42 ┆ ["cutter",    │
│                ┆ 84e53129c7dc8 ┆ "nutmeg",     │
│                ┆ 9f97f2f193384 ┆ "potatoes",   │
│                ┆ 3946745df0613 ┆ "flour",      │
│                ┆ 70bd1ab738bf2 ┆ "white",      │
│                ┆ afe1c537c3    ┆ "black",      │
│                ┆               ┆ "cream",      │
│                ┆               ┆ "butter",     │
│                ┆               ┆ "onion",      │
│                ┆               ┆ "water",      │
│                ┆               ┆ "cookie",     │
│                ┆               ┆ "egg",        │
│                ┆               ┆ "sour",       │
│                

In [ ]:
with pl.Config(fmt_table_cell_list_len=50,
               fmt_str_lengths=2000,
               set_tbl_width_chars=50,
               ):
    print(filtered_to_investigate.select(
        ["title", 
         "mealeon_id",
         "regex_filtered_ingredients"
         ]
         )[50:55]
         )


shape: (5, 3)
┌────────────────┬───────────────┬───────────────┐
│ title          ┆ mealeon_id    ┆ regex_filtere │
│ ---            ┆ ---           ┆ d_ingredients │
│ str            ┆ str           ┆ ---           │
│                ┆               ┆ list[str]     │
╞════════════════╪═══════════════╪═══════════════╡
│ California     ┆ Epicurious-d2 ┆ ["avocado",   │
│ Rolls          ┆ b862c91c84a69 ┆ "sauce",      │
│                ┆ b9b2f2be1a1cc ┆ "scotch",     │
│                ┆ 9702a2d6af8a6 ┆ "crab",       │
│                ┆ f1e5dcaccbacd ┆ "meat",       │
│                ┆ a6ed9f31b0    ┆ "legs",       │
│                ┆               ┆ "wine",       │
│                ┆               ┆ "shelled",    │
│                ┆               ┆ "vinegared",  │
│                ┆               ┆ "water",      │
│                ┆               ┆ "nori",       │
│                ┆               ┆ "soy",        │
│                ┆               ┆ "lemon",      │
│                

In [ ]:
with pl.Config(fmt_table_cell_list_len=50,
               fmt_str_lengths=2000,
               set_tbl_width_chars=50,
               ):
    print(filtered_to_investigate.select(
        ["title", 
         "mealeon_id",
         "regex_filtered_ingredients"
         ]
         )[55:60]
         )


shape: (5, 3)
┌─────────────┬─────────────────┬────────────────┐
│ title       ┆ mealeon_id      ┆ regex_filtered │
│ ---         ┆ ---             ┆ _ingredients   │
│ str         ┆ str             ┆ ---            │
│             ┆                 ┆ list[str]      │
╞═════════════╪═════════════════╪════════════════╡
│ Carnitas    ┆ AllRecipes-9a5d ┆ ["chicken",    │
│             ┆ c3fa2511284deeb ┆ "salsa",       │
│             ┆ 5ee469d110874b0 ┆ "pork",        │
│             ┆ de9c021439cd0e0 ┆ "shoulder",    │
│             ┆ 88d3ef4f53f28bf ┆ "corn",        │
│             ┆                 ┆ "annatto",     │
│             ┆                 ┆ "flour",       │
│             ┆                 ┆ "sazon",       │
│             ┆                 ┆ "water",       │
│             ┆                 ┆ "adobo",       │
│             ┆                 ┆ "pico",        │
│             ┆                 ┆ "pepper",      │
│             ┆                 ┆ "gallo",       │
│             ┆  

In [ ]:
with pl.Config(fmt_table_cell_list_len=50,
               fmt_str_lengths=2000,
               set_tbl_width_chars=50,
               ):
    print(filtered_to_investigate.select(
        ["title", 
         "mealeon_id",
         "regex_filtered_ingredients"
         ]
         )[60:69]
         )


shape: (8, 3)
┌────────────────┬───────────────┬───────────────┐
│ title          ┆ mealeon_id    ┆ regex_filtere │
│ ---            ┆ ---           ┆ d_ingredients │
│ str            ┆ str           ┆ ---           │
│                ┆               ┆ list[str]     │
╞════════════════╪═══════════════╪═══════════════╡
│ Huli Huli      ┆ AllRecipes-1b ┆ ["onions",    │
│ Chicken        ┆ 1ac4f279d0cbb ┆ "chickens",   │
│                ┆ 0c680da7a1839 ┆ "each",       │
│                ┆ d0031a52f3cc7 ┆ "ginger",     │
│                ┆ 3681dbb2644cf ┆ "green",      │
│                ┆ 96d6aec4f9    ┆ "sherry",     │
│                ┆               ┆ "ketchup",    │
│                ┆               ┆ "mustard",    │
│                ┆               ┆ "pineapple",  │
│                ┆               ┆ "garlic",     │
│                ┆               ┆ "soy",        │
│                ┆               ┆ "sauce",      │
│                ┆               ┆ "brown",      │
│                

Need to test search queries that:
1) Get the recipe we're looking for
    - Query should already have the recipe title, recipe cuisine
    - We use the title and cuisine to get the ingredients
2) Do not return the same recipe name back
    eg `WHERE !(title contains [query_recipe_title])`
    or `WHERE !(title [fuzzy logic] [query_recipe_title])`
3) Do not return the same cuisine back
    eg `WHERE !(cuisine contains [query_recipe_cuisine or query_recipe_cuisine_superset])`
4) Maximize ingredient similarity (start with cosine similarity)
    - Could try vespa dotProduct, wand


In [ ]:
# ok baseline recipes
baselines = [
    "BBCFood-11015de4db20e53dcda0035673dfdbeca355d2a1d37d71f13d5fe5ea7ccca02d", #cacio e pepe
    "AllRecipes-fddb273d401ac45b568695daeb1abe0b632696dd867992fd401e31dc794d66f9", #lasagna
    "AllRecipes-715d84d02e183deaaf2400f7526c80a2b01a25032c1abb125c28fe4d879cc672", #margherita pizza
    "AfricanBites-6f62a39913dd8029cd399d4ae47b0551f94a16865a3143b5dd28664c0880596e", #tiramisu
    "AllRecipes-867b4f3b539b381fb9f35252402fc14efc9d6fcc64353089959ac159372a8199", #tiramisu
    "BBCFood-c6b9581aa49ee6b0cc3f686f58cfd124c395284cc46dfcde5bc931982975ee15", #tiramisu
    "BBCFood-d59ab715a7f6f0022155b1149e4e059b0497fd790993fb11f1795b901cae8f97", #cannoli
    "AllRecipes-d431cefd9f595be8a8fcd7ff9e05a82ca1b79220c72edde1aa2a0b6574060c04", #arancini
    "BBCFood-056f016b0b0d8c383c1a6bf47077a46a64529c04ab043c14dce3b23d3980e0ed", # pad thai
    "AllRecipes-d4bb95742a7fd71eb848a7d4b2a4dc13220e5bb378e1b6805d5ea493e0e1166d", # pad thai
    "Epicurious-0293b95bfc74cf90dc07e2ee399312e40e03d14dd99e5db84e678a1393b131cc", # pad thai
    "Epicurious-ca2d5e56121ae04e8b7bdcd07648796b2d95afed017fc56200535ad43ca8e51d", # kung pao chicken
    "Epicurious-7761b0f03aaf56b52b1de417213b66fddf1d49c554dabaacb23c9c7a260982ba", # boeuf bourguignon
    "Epicurious-e367b68ed6df8b0bccbeb73f9d82897294ef267fa3a74d30505ce7470b2384dc", # boeuf bourguignon 
    "AllRecipes-6914cfedf5d3af52ea66433959c958821b8940ee1948fbe29f841d48ad1de4a4", # beef wellington
    "AfricanBites-2f35554786f85a304f1f5e86313f5d35730777a4f0861edf3474a914e9b7ec23", # fish and chips
    "BBCFood-adcbfceaa8636f83ff8866d44bc7a44385c71250d73d427c08dbe345e4127143", # fish and chips
    "BBCFood-0a7c810d7b7c99ae728f9a2deff14340edb099949f20fed75a8083c29ca83aab", # chicken tikka masala
    "Epicurious-a0ab4c8e25f1fe8e65f400f93db23cf55e1e46cb4361cd65426f908f3013f33d", #chicken tikka masala
    "AllRecipes-bb6dd1b85cfd3cbfeb68160c187862ec6de84739123733ceeaae2976141f8936", # yorkshire pudding
    "AllRecipes-92c5c30f5c0b1655a1e1900fcdaa603917f2abf62529c130713412a844046e33", # yorkshire pudding
    "AfricanBites-fc015a5953d8811734f652ad17ba50d73f98d783350df0a95a6e2015007263f0", # apple pie
    "AllRecipes-384fd2f67b06a5bfdf532f94d5e89df99e08b91c5d9dace20fd29bba7b8fdad3", # apple pie, this might need to get taken out
    "Epicurious-aa7806301b6fc904134d692a2a425d0a0fd5a1f6f95cb431e8c31173db7d99fa", # apple pie
    "BBCFood-ee117ce84728c9b2e90654f59895a89dc7c10599b61bed83f55e9bdf723e6890", # scotch egg
    "BBCFood-53dc7bd8da3ef8a7dedfef0b2be1880f71585752d8113313c6d9e49bd5cba6fa", # scotch egg
    "BBCFood-2ab22cd0a3889f38cf740040d33aae5a4baa8568b70cde5233d6f8a673958140", # haggis
    "BBCFood-47410dc7097b2609d5c0a80e4ce6f938b00be18b52907e5a5322e33d6edee335", # buffalo wings
    "AfricanBites-587700762275fbf6a6e2a343b27cc689317725ae2d1724cd33c6c841583afc3e", # banana pudding
    "Epicurious-ca99be627e88806d8357a5cbfdd314b3643028f275e6ae1c90adfcf3ae9e30d4", # cincinnati chili
    "Epicurious-7e9ff3fc13b040941ed9f7a4bbfd809e17a9ce7d5b569f1f6c4ba96e9a9ae140", #kalbi
    "Epicurious-4b6f818877199d375e801cec974c6f6b037b4c0b2aa2bbec0238dd77f797264d", # seafood pancake
    "AllRecipes-c6d0cd77e37ed8069f0d9e72f9e832026459273ac1bc2587800dc8fcd1418ec7", #kimchi soup
    "AllRecipes-48bde26e015dc9cbcbb540c4710248d538227d4afe5aa3e977e34710289789bb", # menudo
    "AllRecipes-cac0d29872bd9f49d34d8c9eac41a3806929420f84e36b8e579c3bce939cdc7b", # pozole
    "AllRecipes-9a5dc3fa2511284deeb5ee469d110874b0de9c021439cd0e088d3ef4f53f28bf", # carnitas
    "BBCFood-6d6cc18dadaffa5ea10194b59aa6e310d34b1f713756d0b528cc57c5e61d92e3", # carnitas
    "AllRecipes-5fc2fc72d7caf510b4ccde625c54a0907de1489d323d385d74c2a6c9c5d61c7b", # elotes
    "AllRecipes-8aa892f3f4d827f781d2bc1491351d982144e2fa2bb3226ebf5cc6af6df5ac5a", # mexican rice
    "AllRecipes-4f31ce862b67de4659f76e3a4de016a32770c878544d112aae95cc09b63d799a", # pho
    "Epicurious-b3f8d44e48bba1b47e5cf1482d8c5e076359fa919061bbfd2f075d131aea2025", # bagels
    "Epicurious-6a7b3ef6808832fbf1ab5a8e539eb794866d630c1b65218ee7b5c385b78f65a4", # yellow pea soup
    "AllRecipes-0caa9e3e35dee2910e6f81c869d1176a2de129b8e3c7fddf6e469b9e3977fadb", # custard 
    "AllRecipes-3c9cbb4d78d017a21a48907adbc31c2593ebbd0717b9391779a2f75b531fcbe4", # tourtiere
    "Epicurious-d2b862c91c84a69b9b2f2be1a1cc9702a2d6af8a6f1e5dcaccbacda6ed9f31b0", # california roll
    "AllRecipes-b8116cc920cd9d916cd196927ad2af1062785cdaa3d3b4fc3e997ee86fd75c84", # pierogi
    "Epicurious-4284e53129c7dc89f97f2f1933843946745df061370bd1ab738bf2afe1c537c3", # pierogi
    "AfricanBites-5ca4ea56e8d49013ebd7457560a386a5e1a1059f267514b7c1dff7b9f0c78d72", # huli huli chicken
    "AllRecipes-bd864502bd1b052f501d3148d66effc966246a69a5e705c585fa366d9599e090", # huli huli chicken
    "AllRecipes-1b1ac4f279d0cbb0c680da7a1839d0031a52f3cc73681dbb2644cf96d6aec4f9", # huli huli chicken
]

In [ ]:
len(baselines)

50

In [ ]:
baseline_df = filtered_to_investigate.filter(
    pl.col("mealeon_id").str.contains_any(baselines)
)

baseline_df

mealeon_id,language,source_id,title,origin,url,ingredients,photo_url,description,steps,cuisines,regex_filtered_ingredients
str,str,str,str,str,str,list[str],str,str,list[str],list[str],list[str]
"""AfricanBites-587700762275fbf6a…","""English""","""597818""","""Banana Pudding""","""AfricanBites""","""https://www.africanbites.com/h…","[""2 ½ tablespoon s (18.75 g) cornstarch "", ""6 tablespoons (75 g) granulated sugar "", … ""homemade whipped cream ""]","""https://www.africanbites.com/w…","""Homemade Banana Pudding – a cl…","[""Add dry ingredients to a saucepan: cornstarch, sugar and salt. Mix to combine. "", ""In a medium bowl, whisk together milk, heavy cream, and egg yolks. Gently whisk wet ingredients into the saucepan making sure there are no lumps or until sugar has dissolved . "", … ""Serve or refrigerate until ready to eat. ""]","[""US Southern""]","[""cream"", ""cornstarch"", … ""wafers""]"
"""AfricanBites-6f62a39913dd8029c…","""English""","""598753""","""Tiramisu""","""AfricanBites""","""https://www.africanbites.com/t…","[""5 large egg yolks , room temperature"", ""¾ cup granulated sugar"", … ""1 cup or more homemade whipped cream (optional, for piping) ""]","""https://www.africanbites.com/w…","""This classic Italian coffee-fl…","[""In a medium saucepan, whisk together egg yolks and sugar until combined. Whisk in milk and cook over medium/low heat stirring constantly. Bring to a boil and boil for 1 minute. Remove from heat and allow to cool. Cover with plastic wrap and refrigerate for 1 hour. "", ""Whisk mascarpone cheese into yolk mixture until smooth. "", … ""Slice or scoop to serve.""]","[""Italian""]","[""liquor"", ""egg"", … ""extract""]"
"""AfricanBites-fc015a5953d881173…","""English""","""607516""","""Apple Pie""","""AfricanBites""","""https://www.africanbites.com/a…","[""Homemade pie crust or store-bought pie crust "", ""7-8 apples (about 3 ½ pounds)"", … ""coarse sugar or turbinado sugar for sprinkling""]","""https://www.africanbites.com/w…","""Made with lightly spiced apple…","[""Start by making the pie crust/and or preparing the pie pan. "", ""Roll out pie crust"", … ""Bake for 40 to 45 minutes or until the pie is bubbly and the crust is golden brown. Remove the foil on the last 15 minutes before the pie has cooked through. Serve warm with ice cream. ""]","[""American""]","[""sprinkling"", ""pie"", … ""egg""]"
"""AfricanBites-2f35554786f85a304…","""English""","""608647""","""Fish and Chips""","""AfricanBites""","""https://www.africanbites.com/f…","[""1 ¼ cup (5 ounces) all-purpose flour"", ""¾ cup (4 ounces) cornstarch"", … ""salt and seasonings to taste""]","""https://www.africanbites.com/w…","""Perfectly seasoned tender fish…","[""In a large bowl, whisk together flour, cornstarch, garlic powder, cayenne, paprika, baking powder, and salt and pepper, to taste. Reserve about ¾ cup of the flour mixture (this is used to dust fish). Set aside until ready to dip the fish. "", ""Pour oil into a cast iron or skillet, enough for deep frying. Preheat oil to about 375 degrees F."", … ""Toss the fries with seasonings or parmesan cheese, if desired, and serve with fish.""]","[""British""]","[""russet"", ""vinegar"", … ""cod""]"
"""AfricanBites-5ca4ea56e8d49013e…","""English""","""567371""","""Huli Huli Chicken""","""AfricanBites""","""https://www.africanbites.com/h…","[""3 1/2- 4 pounds chicken drumsticks"", ""1 cup unsweetened pineapple juice"", … ""Green onions sliced for garnish""]","""https://www.africanbites.com/w…","""Huli Huli Chicken - Grilled or…","[""In a medium bowl, mix the pineapple juice, soy sauce, honey, brown sugar, sriracha, ketchup, ginger, cumin, lemon juice and garlic.""]","[""American""]","[""sriracha"", ""chicken"", … ""green""]"
…,…,…,…,…,…,…,…,…,…,…,…
"""AllRecipes-fddb273d401ac45b568…","""English""","""14054""","""Lasagna""","""AllRecipes""","""https://www.allrecipes.com/rec…","[""¼ cup olive oil"", ""1 onion, chopped"", … ""1 cup grated Parmesan cheese""]","""https://www.allrecipes.com/thm…","

In [ ]:
baseline_df.filter(
    pl.col("title").str.to_lowercase().str.contains("tiramisu")
)

mealeon_id,language,source_id,title,origin,url,ingredients,photo_url,description,steps,cuisines,regex_filtered_ingredients
str,str,str,str,str,str,list[str],str,str,list[str],list[str],list[str]
"""AfricanBites-6f62a39913dd8029c…","""English""","""598753""","""Tiramisu""","""AfricanBites""","""https://www.africanbites.com/t…","[""5 large egg yolks , room temperature"", ""¾ cup granulated sugar"", … ""1 cup or more homemade whipped cream (optional, for piping) ""]","""https://www.africanbites.com/w…","""This classic Italian coffee-fl…","[""In a medium saucepan, whisk together egg yolks and sugar until combined. Whisk in milk and cook over medium/low heat stirring constantly. Bring to a boil and boil for 1 minute. Remove from heat and allow to cool. Cover with plastic wrap and refrigerate for 1 hour. "", ""Whisk mascarpone cheese into yolk mixture until smooth. "", … ""Slice or scoop to serve.""]","[""Italian""]","[""liquor"", ""egg"", … ""extract""]"
"""AllRecipes-867b4f3b539b381fb9f…","""English""","""21412""","""Tiramisu""","""AllRecipes""","""https://www.allrecipes.com/rec…","[""6 large egg yolks"", ""¾ cup white sugar"", … ""1 tablespoon unsweetened cocoa powder""]","""Missing Photo""","""This delicious tiramisu made w…","[""Tiramisu, with its irresistible coffee flavor and lightly sweetened mascarpone, will never go out of style. This tiramisu recipe is a no-bake dessert that's sure to impress even the pickiest of eaters."", ""Tiramisu is a coffee-flavored dessert that features layers of homemade whipped cream, an egg yolk-enriched mascarpone filling, and coffee-soaked ladyfingers."", … ""Enjoy!""]","[""Missing Cuisine""]","[""cookies"", ""egg"", … ""mascarpone""]"
"""BBCFood-c6b9581aa49ee6b0cc3f68…","""English""","""https://www.bbc.co.uk/food/rec…","""Tiramisu""","""BBCFood""","""https://www.bbc.co.uk/food/rec…","[""6 free-range eggs, separated"", ""200g/7oz caster sugar"", … ""cocoa powder, for dusting""]","""https://ichef.bbci.co.uk/food/…","""This authentic Italian tiramis…","[""In a large bowl, whisk the egg yolks and sugar together with an electric whisk until pale and creamy. Mix the mascarpone into the egg mixture until well combined."", ""In a separate bowl, whip the double cream until soft peaks form when the whisk is removed. With a metal spoon, fold the whipped cream into the mascarpone, egg and sugar mixture. "", … ""Dust with cocoa powder before serving.""]","[""Italian""]","[""finger"", ""marsala"", … ""biscuits""]"


In [ ]:
# polars search can be 
recipe_title = "tiramisu"
recipe_cuisine = "italian"

baseline_df.filter(
    pl.col("title").str.to_lowercase().str.contains(recipe_title),
    pl.col("cuisines").list.contains(recipe_cuisine.title())
)


mealeon_id,language,source_id,title,origin,url,ingredients,photo_url,description,steps,cuisines,regex_filtered_ingredients
str,str,str,str,str,str,list[str],str,str,list[str],list[str],list[str]
"""AfricanBites-6f62a39913dd8029c…","""English""","""598753""","""Tiramisu""","""AfricanBites""","""https://www.africanbites.com/t…","[""5 large egg yolks , room temperature"", ""¾ cup granulated sugar"", … ""1 cup or more homemade whipped cream (optional, for piping) ""]","""https://www.africanbites.com/w…","""This classic Italian coffee-fl…","[""In a medium saucepan, whisk together egg yolks and sugar until combined. Whisk in milk and cook over medium/low heat stirring constantly. Bring to a boil and boil for 1 minute. Remove from heat and allow to cool. Cover with plastic wrap and refrigerate for 1 hour. "", ""Whisk mascarpone cheese into yolk mixture until smooth. "", … ""Slice or scoop to serve.""]","[""Italian""]","[""liquor"", ""egg"", … ""extract""]"
"""BBCFood-c6b9581aa49ee6b0cc3f68…","""English""","""https://www.bbc.co.uk/food/rec…","""Tiramisu""","""BBCFood""","""https://www.bbc.co.uk/food/rec…","[""6 free-range eggs, separated"", ""200g/7oz caster sugar"", … ""cocoa powder, for dusting""]","""https://ichef.bbci.co.uk/food/…","""This authentic Italian tiramis…","[""In a large bowl, whisk the egg yolks and sugar together with an electric whisk until pale and creamy. Mix the mascarpone into the egg mixture until well combined."", ""In a separate bowl, whip the double cream until soft peaks form when the whisk is removed. With a metal spoon, fold the whipped cream into the mascarpone, egg and sugar mixture. "", … ""Dust with cocoa powder before serving.""]","[""Italian""]","[""finger"", ""marsala"", … ""biscuits""]"


- From this dataframe, we want the `mealeon_id`, `ingredients` to be used in Vespa

- `mealeon_id` can be used directly in vespa. JUST KIDDING

- `title`

-  `ingredients` need some form of vectorization/embedding creation

- embedding creation can be done with vespa feed?
  - might be able to use json feed 

- vespa distance-metric with the filters above for avoiding same name and same cuisine

In [ ]:
with app.syncio(connections=1) as session:
    response: VespaQueryResponse = session.query(
        yql=f"select * from sources mealeon3 where title contains '{recipe_title}' limit 10",
        query=recipe_title,
        ranking="bm25"
        # body={"input.query(q)": f"embed({query})"},
    )
    assert response.is_successful()

In [ ]:
pprint.pp(response.hits)

[]


In [ ]:
recipe_title

'tiramisu'

In [ ]:
with app.syncio(connections=1) as session:
    response: VespaQueryResponse = session.query(
        yql=f'select * from sources mealeon3 where !(title contains "{recipe_title}") and !(cuisine in {recipe_cuisine}) limit 10',
        query=query,
        ranking="bm25"
        # body={"input.query(q)": f"embed({query})"},
    )
    assert response.is_successful()


In [ ]:
with app.syncio(connections=1) as session:
    response: VespaQueryResponse = session.query(
        yql=f'select * from sources mealeon3 where !(cuisine in {recipe_cuisine}) limit 10',
        query=query,
        ranking="bm25"
        # body={"input.query(q)": f"embed({query})"},
    )
    assert response.is_successful()


In [ ]:
type(recipe_cuisine)

In [ ]:
with app.syncio(connections=1) as session:
    response: VespaQueryResponse = session.query(
        yql=f"select * from sources mealeon3 where !(cuisine in {[recipe_cuisine]}) limit 10",
        query=query,
        ranking="bm25"
        # body={"input.query(q)": f"embed({query})"},
    )
    assert response.is_successful()


In [ ]:
recipe_cuisine

'italian'

In [ ]:
print([recipe_cuisine])

In [ ]:
cuisines_queried = [recipe_cuisine]

with app.syncio(connections=1) as session:
    response: VespaQueryResponse = session.query(
        yql=f"select * from sources mealeon3 where !(cuisine in {cuisines_queried}) limit 10",
        query=query,
        ranking="bm25"
        # body={"input.query(q)": f"embed({query})"},
    )
    assert response.is_successful()


In [ ]:
with app.syncio(connections=1) as session:
    response: VespaQueryResponse = session.query(
        yql=f"select * from sources mealeon3 where !(cuisine in ({recipe_cuisine}) limit 10",
        query=query,
        ranking="bm25"
        # body={"input.query(q)": f"embed({query})"},
    )
    assert response.is_successful()


In [ ]:
with app.syncio(connections=1) as session:
    response: VespaQueryResponse = session.query(
        yql=f"select * from sources mealeon3 where !(cuisines in ('italian')) limit 10",
        query=query,
        ranking="bm25"
        # body={"input.query(q)": f"embed({query})"},
    )
    assert response.is_successful()


In [ ]:
pprint.pp(response.hits)

[]


In [ ]:
# actual results
results = resp_json['hits']
results

In [ ]:
# | hide
nbdev.nbdev_export()